# AASTU Student Handbook RAG Assistant
### End-to-End Retrieval-Augmented Generation with ChromaDB & Google Gemini
**Week 4: Generative AI & AI Applications**

This notebook implements a production-grade, grounded RAG assistant for the Addis Ababa Science and Technology University (AASTU) Student Handbook. It features continuous cross-page chunking, local ChromaDB dense semantic retrieval, an externalized system prompt with strict anti-hallucination guardrails, Pydantic structured output validation, and an automated regression test suite.

## 1. Environment & Dependencies Setup

In [ ]:
# Install required dependencies for Google Colab / local environment
!pip install -q pypdf chromadb requests python-dotenv pydantic

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from typing import List, Dict, Any
from dotenv import load_dotenv

# Load API Key from environment or local .env
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY') or os.getenv('APIKEY')
if not api_key:
    api_key = input('Please enter your Gemini API key: ').strip()
print(f"API Key configured: {api_key[:6]}...{api_key[-4:]}")

## 2. Phase 1: PDF Ingestion & Continuous Document Chunking

Instead of page-isolated chunking (which misses sentences spanning across page boundaries), we use **continuous document-level chunking**:
1. Extract text from all 375 pages with page boundary word offsets.
2. Create rolling windows of **350 words** with a **50-word sliding overlap**.
3. Map each chunk back to its exact source page span (e.g., `Pages 25–27`).

In [ ]:
# Import ingestion logic
from ingestion import load_and_extract_pdf, create_continuous_chunks, CHUNKS_PATH

if CHUNKS_PATH.exists():
    with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    print(f"Loaded {len(chunks)} pre-computed continuous chunks from {CHUNKS_PATH.name}")
else:
    pages = load_and_extract_pdf()
    chunks = create_continuous_chunks(pages)

# Inspect sample chunk
sample = chunks[15]
print(f"Sample Chunk ID:     {sample['chunk_id']}")
print(f"Page Span:           {sample['citation_label']} (Pages {sample['pages']})")
print(f"Word Count:          {sample['word_count']} words")
print(f"Text Preview:        {sample['text'][:180]}...")

## 3. Phase 2: ChromaDB Dense Semantic Vector Store

Chunks are embedded into a local ChromaDB collection using the `all-MiniLM-L6-v2` dense embedding model with cosine similarity.

In [ ]:
from retrieval import get_chroma_client, get_or_create_collection, index_chunks, retrieve_context

client = get_chroma_client()
collection = get_or_create_collection(client)
total_indexed = index_chunks(collection)
print(f"Total chunks indexed: {total_indexed}")

# Test semantic retrieval
test_query = "What are the criteria for academic probation or dismissal?"
hits = retrieve_context(test_query, top_k=3)

print(f"\nTop 3 hits for query: '{test_query}'")
for i, hit in enumerate(hits, 1):
    print(f"  [{i}] {hit['citation_label']} (Cosine Distance: {hit['distance']:.3f})")
    print(f"      Preview: {hit['text'][:120]}...")

## 4. Phase 3: Prompt Engineering & Grounded Generation Engine

We enforce strict grounding rules via external prompt template (`prompts/qa_system.txt`) and guarantee schema validity using Pydantic.

In [ ]:
from rag_engine import ask_handbook, load_prompt_template

prompt_preview = load_prompt_template()
print("=== System Prompt Preview (First 400 chars) ===")
print(prompt_preview[:400] + "...")

# Run grounded in-scope question
response = ask_handbook("What is the policy for late registration?", top_k=3, api_key=api_key)

print("\n=== Grounded Answer ===")
print(response['answer'])
print(f"Citations:  {response['citations']}")
print(f"Confidence: {response['confidence']}")
print(f"Can Answer: {response['can_answer']}")

## 5. Anti-Hallucination & Adversarial Query Test

When asked a question completely outside the handbook (e.g., cooking or geography), the assistant must safely refuse rather than fabricate an answer.

In [ ]:
adversarial_query = "How do I bake a chocolate cake at home?"
adv_response = ask_handbook(adversarial_query, top_k=2, api_key=api_key)

print(f"Query:      {adversarial_query}")
print(f"Answer:     {adv_response['answer']}")
print(f"Citations:  {adv_response['citations']} (Expected: Empty)")
print(f"Can Answer: {adv_response['can_answer']} (Expected: False - Refusal)")

## 6. Phase 4: Golden Regression Evaluation Suite

Evaluates 8 curated test cases (6 in-scope across different academic domains + 2 adversarial out-of-scope) measuring:
- **Retrieval Hit Rate**
- **Grounding Decision Accuracy**
- **Adversarial Refusal Rate**
- **Citation Correctness**

In [ ]:
from evaluate import run_evaluation

summary = run_evaluation(verbose=True)
print(f"\nFinal Pass Rate: {summary['overall_pass_rate']}% ({summary['passed_tests']}/{summary['total_tests']})")